In [6]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV


from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix)

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report
from imblearn.under_sampling import RandomUnderSampler


In [2]:
TW_500= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=500/Twitter-Relative-Sigma-500.data",
    sep=",",
    header=None
)

groups = ["NCD", 'AI', 'AS(NA)', 'BL',
         'NAC', 'AS(NAC)', 'CS', 'AT', 'NA','ADL', 'NAD']

columns = []
for group in groups:
    for t in range(7):
        columns.append(f"{group}_{t}")

columns.append("label") 

TW_500.columns = columns

In [3]:
top_features = ['NCD','NAD','NAC','AS(NAC)','NA', 'label']

selected_columns = [
    col for col in TW_500.columns 
    if any(col.startswith(f + '_') for f in top_features)
]

Selected_500 = TW_500[selected_columns]
Selected_500['label']=TW_500['label']

print("Number of features:", len(selected_columns))
Selected_500.columns


Number of features: 35


Index(['NCD_0', 'NCD_1', 'NCD_2', 'NCD_3', 'NCD_4', 'NCD_5', 'NCD_6', 'NAC_0',
       'NAC_1', 'NAC_2', 'NAC_3', 'NAC_4', 'NAC_5', 'NAC_6', 'AS(NAC)_0',
       'AS(NAC)_1', 'AS(NAC)_2', 'AS(NAC)_3', 'AS(NAC)_4', 'AS(NAC)_5',
       'AS(NAC)_6', 'NA_0', 'NA_1', 'NA_2', 'NA_3', 'NA_4', 'NA_5', 'NA_6',
       'NAD_0', 'NAD_1', 'NAD_2', 'NAD_3', 'NAD_4', 'NAD_5', 'NAD_6', 'label'],
      dtype='object')

In [4]:
X = Selected_500.drop(columns=['label'])
y = Selected_500['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [7]:

X = TW_500.drop(columns=['label'])
y = TW_500['label']

# --- 2. split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- 3. ratios to test ---
ratios = {
    "1:10": 0.1,
    "1:15": 0.067,
    "1:20": 0.05,
    "1:25": 0.04,
    "1:30": 0.033
}

results = {}

for name, ratio in ratios.items():
    print(f"\n===== Undersampling {name} =====")

    # undersampling
    rus = RandomUnderSampler(sampling_strategy=ratio, random_state=42)
    X_res, y_res = rus.fit_resample(X_train, y_train)

    print("After:", y_res.value_counts())

    # model
    rf = RandomForestClassifier(random_state=42)

    param_grid = {
        'n_estimators': [100],
        'max_depth': [20, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    }

    grid = GridSearchCV(
        rf,
        param_grid,
        scoring='f1',
        cv=3,
        n_jobs=-1
    )

    grid.fit(X_res, y_res)

    best_model = grid.best_estimator_

    # prediction
    y_pred = best_model.predict(X_test)

    f1 = f1_score(y_test, y_pred)
    results[name] = f1

    print("Best params:", grid.best_params_)
    print("F1:", f1)

# --- 4. summary ---
print("\n===== FINAL RESULTS =====")
for k, v in results.items():
    print(f"{k}: {v:.4f}")


===== Undersampling 1:10 =====
After: label
0.0    28960
1.0     2896
Name: count, dtype: int64
Best params: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
F1: 0.5849387040280211

===== Undersampling 1:15 =====
After: label
0.0    43223
1.0     2896
Name: count, dtype: int64
Best params: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
F1: 0.6059379217273954

===== Undersampling 1:20 =====
After: label
0.0    57920
1.0     2896
Name: count, dtype: int64
Best params: {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
F1: 0.6230483271375464

===== Undersampling 1:25 =====
After: label
0.0    72400
1.0     2896
Name: count, dtype: int64
Best params: {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
F1: 0.6290196078431373

===== Undersampling 1:30 =====
After: label
0.0    87757
1.0     2896
Name: count, dtype: int64
Best params: {'max_dept